In [0]:
# ============================================================
# GOLD LAYER - Business Ready Aggregations
# Objective 1: Restaurant Revenue & Cuisine Trends
# Objective 2: Customer Spending & Budget Behavior
# ============================================================

# ===== CELL 1: Drop Existing Gold Tables =====

spark.sql("DROP TABLE IF EXISTS gold.revenue_by_cuisine")
spark.sql("DROP TABLE IF EXISTS gold.revenue_by_restaurant")
spark.sql("DROP TABLE IF EXISTS gold.revenue_by_city")
spark.sql("DROP TABLE IF EXISTS gold.top_spenders")
spark.sql("DROP TABLE IF EXISTS gold.spending_by_city")

print("✅ Old Gold tables dropped!")

# ===== CELL 2: Load Silver Tables =====

from pyspark.sql.functions import (
    col, sum as spark_sum, count, avg,
    round as spark_round, when
)

silver_orders = spark.table("silver.orders_enriched")
silver_order_details = spark.table("silver.order_details_enriched")
silver_spending = spark.table("silver.member_spending")

print("✅ Silver tables loaded!")

# ===== CELL 3: OBJECTIVE 1 — Revenue by Cuisine =====

revenue_by_cuisine = (
    silver_orders
    .groupBy("restaurant_type")
    .agg(
        count("order_id").alias("total_orders"),
        spark_round(spark_sum("total_order"), 2).alias("total_revenue"),
        spark_round(spark_sum("commission_earned"), 2).alias("total_commission"),
        spark_round(avg("total_order"), 2).alias("avg_order_value")
    )
    .orderBy(col("total_commission").desc())
)

revenue_by_cuisine.write.format("delta").mode("overwrite").saveAsTable("gold.revenue_by_cuisine")
print("✅ Gold: revenue_by_cuisine saved!")
revenue_by_cuisine.display()

# ===== CELL 4: OBJECTIVE 1 — Revenue by Restaurant =====

revenue_by_restaurant = (
    silver_orders
    .groupBy("restaurant_name", "restaurant_type", "restaurant_city")
    .agg(
        count("order_id").alias("total_orders"),
        spark_round(spark_sum("total_order"), 2).alias("total_revenue"),
        spark_round(spark_sum("commission_earned"), 2).alias("total_commission"),
        spark_round(avg("total_order"), 2).alias("avg_order_value")
    )
    .orderBy(col("total_revenue").desc())
)

revenue_by_restaurant.write.format("delta").mode("overwrite").saveAsTable("gold.revenue_by_restaurant")
print("✅ Gold: revenue_by_restaurant saved!")
revenue_by_restaurant.display()

# ===== CELL 5: OBJECTIVE 1 — Revenue by City =====

revenue_by_city = (
    silver_orders
    .groupBy("restaurant_city")
    .agg(
        count("order_id").alias("total_orders"),
        spark_round(spark_sum("total_order"), 2).alias("total_revenue"),
        spark_round(spark_sum("commission_earned"), 2).alias("total_commission")
    )
    .orderBy(col("total_revenue").desc())
)

revenue_by_city.write.format("delta").mode("overwrite").saveAsTable("gold.revenue_by_city")
print("✅ Gold: revenue_by_city saved!")
revenue_by_city.display()

# ===== CELL 6: OBJECTIVE 2 — Top Spenders =====

top_spenders = (
    silver_spending
    .groupBy("member_id", "first_name", "surname", "city")
    .agg(
        spark_round(spark_sum("total_expense"), 2).alias("lifetime_expense"),
        spark_round(avg("spending_ratio"), 2).alias("avg_spending_ratio"),
        spark_sum("is_overspending").alias("months_overspent")
    )
    .orderBy(col("lifetime_expense").desc())
)

top_spenders.write.format("delta").mode("overwrite").saveAsTable("gold.top_spenders")
print("✅ Gold: top_spenders saved!")
top_spenders.display()

# ===== CELL 7: OBJECTIVE 2 — Spending by City =====

spending_by_city = (
    silver_spending
    .groupBy("city")
    .agg(
        spark_round(spark_sum("total_expense"), 2).alias("total_spending"),
        spark_round(avg("monthly_budget"), 2).alias("avg_budget"),
        spark_round(avg("spending_ratio"), 2).alias("avg_spending_ratio"),
        spark_sum("is_overspending").alias("overspend_count")
    )
    .orderBy(col("total_spending").desc())
)

spending_by_city.write.format("delta").mode("overwrite").saveAsTable("gold.spending_by_city")
print("✅ Gold: spending_by_city saved!")
spending_by_city.display()

# ===== CELL 8: Verify All Gold Tables =====

print("\n=== ALL GOLD TABLES ===")
spark.sql("SHOW TABLES IN gold").display()

print("\n=== OBJECTIVE 1: Revenue by Cuisine ===")
spark.sql("SELECT * FROM gold.revenue_by_cuisine ORDER BY total_commission DESC").display()

print("\n=== OBJECTIVE 1: Top 10 Restaurants by Revenue ===")
spark.sql("SELECT * FROM gold.revenue_by_restaurant ORDER BY total_revenue DESC LIMIT 10").display()

print("\n=== OBJECTIVE 2: Top 10 Spenders ===")
spark.sql("SELECT * FROM gold.top_spenders LIMIT 10").display()

print("\n=== OBJECTIVE 2: Spending by City ===")
spark.sql("SELECT * FROM gold.spending_by_city").display()

✅ Old Gold tables dropped!
✅ Silver tables loaded!
✅ Gold: revenue_by_cuisine saved!


restaurant_type,total_orders,total_revenue,total_commission,avg_order_value
Fast Food,4028,344354.38,257.60,85.49
Asian,2780,230071.49,172.85,82.76
Indian,2357,213700.25,170.48,90.67
Homemade,1201,124596.49,113.72,103.74
Italian,1635,151329.95,101.16,92.56


✅ Gold: revenue_by_restaurant saved!


restaurant_name,restaurant_type,restaurant_city,total_orders,total_revenue,total_commission,avg_order_value
Restaurant 10,Homemade,Ramat Gan,404,44251.52,44.26,109.53
Restaurant 18,Italian,Ramat Hasharon,443,43457.40,21.95,98.10
Restaurant 11,Indian,Herzelia,394,42690.39,32.07,108.35
Restaurant 30,Homemade,Tel Aviv,427,42625.11,31.90,99.82
Restaurant 20,Fast Food,Ramat Gan,414,40605.60,30.50,98.08
Restaurant 1,Italian,Ramat Hasharon,400,39749.56,29.65,99.37
Restaurant 16,Fast Food,Ramat Hasharon,413,39559.19,29.84,95.78
Restaurant 13,Indian,Ramat Hasharon,419,39479.48,29.81,94.22
Restaurant 8,Asian,Givatayim,392,38836.34,29.59,99.07
Restaurant 17,Fast Food,Ramat Gan,421,37916.01,19.02,90.06


✅ Gold: revenue_by_city saved!


...

restaurant_name,restaurant_type,restaurant_city,total_orders,total_revenue,total_commission,avg_order_value
Restaurant 10,Homemade,Ramat Gan,404,44251.52,44.26,109.53
Restaurant 18,Italian,Ramat Hasharon,443,43457.40,21.95,98.10
Restaurant 11,Indian,Herzelia,394,42690.39,32.07,108.35
Restaurant 30,Homemade,Tel Aviv,427,42625.11,31.90,99.82
Restaurant 20,Fast Food,Ramat Gan,414,40605.60,30.50,98.08
Restaurant 1,Italian,Ramat Hasharon,400,39749.56,29.65,99.37
Restaurant 16,Fast Food,Ramat Hasharon,413,39559.19,29.84,95.78
Restaurant 13,Indian,Ramat Hasharon,419,39479.48,29.81,94.22
Restaurant 8,Asian,Givatayim,392,38836.34,29.59,99.07
Restaurant 17,Fast Food,Ramat Gan,421,37916.01,19.02,90.06



=== OBJECTIVE 2: Top 10 Spenders ===


member_id,first_name,surname,city,lifetime_expense,avg_spending_ratio,months_overspent
31,Kaisha,Watkins,Ramat Gan,6000.0,0.4,0
180,Bilaal,Berry,Ramat Gan,6000.0,0.35,0
192,Amrit,Haworth,Givatayim,6000.0,0.45,0
187,Niyah,Whelan,Ramat Hasharon,6000.0,0.38,0
42,Aahil,Redman,Ramat Hasharon,6000.0,0.35,0
91,Leo,Walton,Givatayim,6000.0,0.44,0
158,Kristi,Faulkner,Givatayim,6000.0,0.38,0
106,Simeon,Guevara,Ramat Hasharon,6000.0,0.53,0
1,Ollie,Kinney,Herzelia,6000.0,0.45,0
46,Mariyah,Green,Herzelia,6000.0,0.39,0



=== OBJECTIVE 2: Spending by City ===


city,total_spending,avg_budget,avg_spending_ratio,overspend_count
Ramat Hasharon,202200.0,2614.5,0.32,1
Herzelia,183000.0,2653.4,0.25,0
Givatayim,171600.0,2624.05,0.27,0
Tel Aviv,162000.0,2606.75,0.29,0
Ramat Gan,129000.0,2622.93,0.3,0
